# ADNI MCI Stability Classifier — Training & Saving Pipeline
## RNN cfg2 (hidden=256, layers=2) + By-Domain PCA | Full Stream & Algerian Stream

This notebook:
1. Loads and preprocesses the ADNI stability dataset (exact pipeline from dim-red notebooks)
2. Applies **by-domain PCA** dimensionality reduction (6 components per clinical domain)
3. Trains **RNN cfg2** (`RecurrentClassifier`, cell_type='rnn', hidden=256, layers=2) for Full and Algerian streams
4. **Saves** models + all preprocessing artifacts (PCA reducers, scalers, encoders, feature lists) for local inference
5. Provides **`predict_patient_sequence()`** helpers for longitudinal inference on new patients

**Target**: 3-class stability label — `CN` / `MCI_stable` / `MCI_converting`


## 0. Imports & Config

In [1]:
import warnings, random, time, copy, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ── Training budget ──────────────────────────────────────────────
EPOCHS   = 100
PATIENCE = 8
N_PER_DOMAIN = 6   # PCA components kept per clinical domain

# ── RNN cfg2 hyperparameters (Config B: wider hidden) ────────────
# RNN cfg2: CORRECT config from dim-red experiments (hidden=128, layers=3, batch=16)
RNN_CFG2 = {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.3, 'lr': 3e-4, 'batch_size': 16}

# ── Output directory ─────────────────────────────────────────────
SAVE_ROOT = './stability_models'
os.makedirs(f'{SAVE_ROOT}/full', exist_ok=True)
os.makedirs(f'{SAVE_ROOT}/alg',  exist_ok=True)
print(f'Models will be saved to: {SAVE_ROOT}/')


Device: cpu
Models will be saved to: ./stability_models/


## 1. Data Paths — Update These

In [2]:
DATA_PATH = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/adni_stability.csv'
MTA_PATH  = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/MTA_labels_final.csv'
AMY_PATH  = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/amy_dataset.csv'


## 2. Feature Definitions

In [3]:
TARGET_COL = 'STABILITY_LABEL'

LEAKING_COLS = [
    'DIAGNOSIS', 'DIAGNOSIS_LABEL', 'TRAJECTORY_CLEANED',
    'DXMDUE', 'DXDEP', 'DXCONFID', 'OBS_SPAN_YEARS', 'DXDSEV'
]

COLS_ALG = [
    "RID", "VISCODE2", "PTGENDER", "age", "PTHAND", "PTMARRY", "PTEDUCAT", "PTWORK", "PTNOTRT",
    "VISDATE", "MMDATE", "MMYEAR", "MMMONTH", "MMDAY", "MMREAD", "MMWRITE", "MMDRAW", "MMREPEAT",
    "MMSEASON", "MMHOSPIT", "MMFLOOR", "MMCITY", "WORD1", "WORD2", "WORD3", "MMSCORE", "MOCA", "CUBE",
    "CLOCKCON", "CLOCKNO", "CLOCKHAN", "DIGFOR", "DIGBACK", "SERIAL1", "SERIAL2", "SERIAL3", "SERIAL4",
    "SERIAL5", "REPEAT1", "REPEAT2", "FFLUENCY", "FAQFORM", "FAQFINAN", "FAQSHOP", "FAQGAME", "FAQBEVG",
    "FAQMEAL", "FAQEVENT", "FAQTV", "FAQREM", "FAQTRAVL", "FAQ",
    "Creatinine", "Calcium", "Direct Bilirubin", "Platelet Ct.", "Red Blood Cell Count",
    "Thyroid-stimulating hormone", "Total Bilirubin", "glucose", "Vitamin B12", "White Blood Cell Count",
    "Abeta42", "Abeta40", "Abeta_ratio", "Homocysteine", "Hemoglobin A1C",
    "Tau181", "pTau181", "Gamma-Glutamyltransferase", "Hematocrit", "Hemoglobin",
    "MOTHDEM", "MOTHAD", "MOTHSXAGE", "FATHDEM", "FATHAD", "FATHSXAGE", "SIBGENDER", "SIBDEMENT", "SIBAD",
    "SIBSXAGE", "IHSYMPTOM", "IHDESC", "IHCHRON", "IHSEVER", "IHPRESENT", "IHSURG", "MH19OTHR", "MHPSYCH",
    "MH2NEURL", "MH3HEAD", "MH4CARD", "MH5RESP", "MH6HEPAT", "MH7DERM", "MH8MUSCL", "MH9ENDO",
    "MH14BALCH", "MH14CALCH", "MH10GAST", "MH11HEMA", "MH12RENA", "MH13ALLE", "MH14ALCH", "MH14AALCH",
    "MH17MALI", "MH18SURG", "MH15DRUG", "MH15ADRUG", "MH15BDRUG",
    "MH16SMOK", "MH16ASMOK", "MH16BSMOK", "MH16CSMOK",
    "BSXSYMNO", "BSXSEVER", "BSXCHRON", "KEYMED", "CMMED", "CMDOSE", "CMREASON",
    'AMYLOID_STATUS', 'MTA_ATROPHY', TARGET_COL,
]

DELTA_COLS = [
    "FAQFORM","FAQFINAN","FAQSHOP","FAQGAME","FAQBEVG","FAQMEAL",
    "FAQEVENT","FAQTV","FAQREM","FAQTRAVL","FAQ",
    "MMSCORE","MOCA",
    "Abeta42","Abeta40","Abeta_ratio",
    "pTau217","npTau217","Tau181","pTau181",
    "CUBE","CLOCKCON","CLOCKNO","CLOCKHAN", 'AMYLOID_STATUS', 'MTA_ATROPHY'
]

LABEL_MAP_3 = {
    'CN':           'CN',
    'MCI stable':   'MCI_stable',
    'MCI unstable': 'MCI_converting',
    'Dementia':     'MCI_converting',
    'MCI':          None,
}

DOMAIN_GROUPS = {
    'cognitive': [
        'MMDATE','MMYEAR','MMMONTH','MMDAY','MMREAD','MMWRITE','MMDRAW',
        'MMREPEAT','MMSEASON','MMHOSPIT','MMFLOOR','MMCITY',
        'WORD1','WORD2','WORD3','MMSCORE','MOCA','CUBE',
        'CLOCKCON','CLOCKNO','CLOCKHAN','DIGFOR','DIGBACK',
        'SERIAL1','SERIAL2','SERIAL3','SERIAL4','SERIAL5',
        'REPEAT1','REPEAT2','FFLUENCY',
    ],
    'functional': [
        'FAQFORM','FAQFINAN','FAQSHOP','FAQGAME','FAQBEVG',
        'FAQMEAL','FAQEVENT','FAQTV','FAQREM','FAQTRAVL','FAQ',
    ],
    'biomarker': [
        'Abeta42','Abeta40','Abeta_ratio','Tau181','pTau181','Homocysteine','Hemoglobin A1C',
        'Creatinine','Calcium','Direct Bilirubin','Total Bilirubin',
        'Platelet Ct.','Red Blood Cell Count','Thyroid-stimulating hormone',
        'glucose','Vitamin B12','White Blood Cell Count',
        'Gamma-Glutamyltransferase','Hematocrit','Hemoglobin',
        'AMYLOID_STATUS','MTA_ATROPHY',
    ],
    'demographics': [
        'PTGENDER','age','PTHAND','PTMARRY','PTEDUCAT','PTWORK','PTNOTRT',
    ],
    'medical_history': [
        'MOTHDEM','MOTHAD','MOTHSXAGE','FATHDEM','FATHAD','FATHSXAGE',
        'SIBGENDER','SIBDEMENT','SIBAD','SIBSXAGE',
        'IHSYMPTOM','IHDESC','IHCHRON','IHSEVER','IHPRESENT','IHSURG',
        'MH2NEURL','MH3HEAD','MH4CARD','MH5RESP','MH6HEPAT','MH7DERM',
        'MH8MUSCL','MH9ENDO','MH10GAST','MH11HEMA','MH12RENA','MH13ALLE',
        'MH14ALCH','MH15DRUG','MH16SMOK','MH17MALI','MH18SURG','MHPSYCH',
        'BSXSYMNO','BSXSEVER','BSXCHRON',
    ],
}

print(f"Domain groups: {list(DOMAIN_GROUPS.keys())}")
print(f"Algerian cols (before intersection): {len(COLS_ALG)}")


Domain groups: ['cognitive', 'functional', 'biomarker', 'demographics', 'medical_history']
Algerian cols (before intersection): 125


## 3. Load & Merge Data

In [4]:
raw = pd.read_csv(DATA_PATH, low_memory=False)

mta = pd.read_csv(MTA_PATH)
amy = pd.read_csv(AMY_PATH)
raw = raw.merge(mta[['RID','VISCODE2','MTA_ATROPHY']].drop_duplicates(['RID','VISCODE2']),
                on=['RID','VISCODE2'], how='left')
raw = raw.merge(amy[['RID','VISCODE2','AMYLOID_STATUS']].drop_duplicates(['RID','VISCODE2']),
                on=['RID','VISCODE2'], how='left')

print(f'Raw shape: {raw.shape}')
print(f'Patients: {raw["RID"].nunique()}')
print(f'\n{TARGET_COL} distribution:')
print(raw[TARGET_COL].value_counts())


Raw shape: (22071, 148)
Patients: 4798

STABILITY_LABEL distribution:
STABILITY_LABEL
CN              6583
MCI stable      5657
MCI unstable    4589
Dementia        2293
MCI             1926
Name: count, dtype: int64


## 4. Preprocessing Pipeline

In [5]:
def viscode_to_month(vc):
    """Convert VISCODE2 strings to numeric month numbers."""
    if pd.isna(vc): return np.nan
    vc = str(vc).strip().lower()
    if vc in ('sc', 'f', 'uns'): return -1
    if vc in ('bl', 'v01'):      return 0
    if vc.startswith('m'):
        try: return int(vc[1:])
        except: return np.nan
    return np.nan


def preprocess(df_raw, feature_mode='full', label_mode='3class'):
    """
    Full preprocessing pipeline.
    feature_mode: 'full' | 'algerian'
    label_mode  : '3class' → CN / MCI_stable / MCI_converting
    Returns: (df, feature_cols, le_target, cat_le_map, global_means_series)
    The extra objects are saved for inference.
    """
    df = df_raw.copy()

    # 3-class target remapping
    if label_mode == '3class':
        df[TARGET_COL] = df[TARGET_COL].map(LABEL_MAP_3)
        n_before = df['RID'].nunique()
        df = df[df[TARGET_COL].notna()]
        print(f'  Dropped {n_before - df["RID"].nunique()} patients with undetermined MCI label')

    # Month encoding & sort
    df['_month'] = df['VISCODE2'].apply(viscode_to_month)
    df = df.sort_values(['RID', '_month'])

    # Trim Dementia visits for converting patients
    def trim_dementia_rows(grp):
        stab = grp[TARGET_COL].iloc[0]
        if stab in ('MCI unstable', 'MCI_converting'):
            dem_mask = (grp.get('DIAGNOSIS_LABEL', pd.Series(dtype=str)) == 'Dementia')
            if dem_mask.any():
                grp = grp.iloc[:dem_mask.values.argmax()]
        return grp

    df = df.groupby('RID', group_keys=False).apply(trim_dementia_rows)
    valid = df.groupby('RID').size()
    df    = df[df['RID'].isin(valid[valid >= 1].index)]
    print(f'After dementia trimming: {df.shape}')

    # Feature selection
    if feature_mode == 'algerian':
        keep = [c for c in COLS_ALG if c in df.columns]
        df   = df[list(set(keep + ['RID', '_month', 'DIAGNOSIS_LABEL']))]
    else:
        drop = [c for c in LEAKING_COLS if c in df.columns and c != TARGET_COL]
        df   = df.drop(columns=drop, errors='ignore')

    # VISCODE2 → numeric month
    if 'VISCODE2' in df.columns:
        df['VISCODE2_num'] = df['_month']
        df = df.drop(columns=['VISCODE2'])

    # VISDATE → days since first visit
    if 'VISDATE' in df.columns:
        df['VISDATE'] = pd.to_datetime(df['VISDATE'], errors='coerce')
        df['VISDATE_days'] = df.groupby('RID')['VISDATE'].transform(
            lambda x: (x - x.min()).dt.days)
        df = df.drop(columns=['VISDATE'])

    # Categorical encoding
    ignore  = {'RID', TARGET_COL}
    cat_cols = [c for c in df.select_dtypes(include=['object', 'string']).columns if c not in ignore]
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ignore]

    cat_le_map = {}
    for c in cat_cols:
        mode_val = df[c].mode(dropna=True)
        mode_val = mode_val.iloc[0] if len(mode_val) else 'UNKNOWN'
        df[c] = df[c].fillna(mode_val)
        le = LabelEncoder()
        df[c] = le.fit_transform(df[c].astype(str))
        cat_le_map[c] = le
    num_cols = num_cols + cat_cols

    # Per-patient mean imputation → global mean fallback
    df[num_cols] = df.groupby('RID')[num_cols].transform(lambda x: x.fillna(x.mean()))
    global_means = df[num_cols].mean()
    df[num_cols] = df[num_cols].fillna(global_means)

    # Missing indicator flags
    flag_cols = []
    for c in num_cols:
        if c in df_raw.columns and df_raw.loc[df.index, c].isnull().any():
            flag_name = f'{c}_missing'
            df[flag_name] = df_raw.loc[df.index, c].isnull().astype(int).values
            flag_cols.append(flag_name)

    # Delta (change) columns
    delta_present = [c for c in DELTA_COLS if c in df.columns]
    for c in delta_present:
        df[f'{c}_delta'] = df.groupby('RID')[c].diff().fillna(0)

    # Final feature list
    feature_cols = [c for c in df.columns
                    if c not in (ignore | {'_month', 'TRAJECTORY_CLEANED', 'OBS_SPAN_YEARS',
                                           'DIAGNOSIS_LABEL', 'DIAGNOSIS'})
                    and c in df.select_dtypes(include=[np.number]).columns]

    # Encode target
    df = df[df[TARGET_COL].notna()].copy()
    le_target = LabelEncoder()
    df['target'] = le_target.fit_transform(df[TARGET_COL].astype(str))

    print(f'Feature columns: {len(feature_cols)}')
    print(f'Target classes: {list(le_target.classes_)}')
    return df, feature_cols, le_target, cat_le_map, global_means[num_cols]

print('=== FULL FEATURE SET — 3-class target ===')
df_full, feat_full, le_full, cat_le_full, gmeans_full = preprocess(raw, 'full', '3class')

print('\n=== ALGERIAN FEATURE SET — 3-class target ===')
df_alg, feat_alg, le_alg, cat_le_alg, gmeans_alg = preprocess(raw, 'algerian', '3class')


=== FULL FEATURE SET — 3-class target ===
  Dropped 1815 patients with undetermined MCI label
After dementia trimming: (14934, 149)
Feature columns: 300
Target classes: ['CN', 'MCI_converting', 'MCI_stable']

=== ALGERIAN FEATURE SET — 3-class target ===
  Dropped 1815 patients with undetermined MCI label
After dementia trimming: (14934, 149)
Feature columns: 269
Target classes: ['CN', 'MCI_converting', 'MCI_stable']


## 5. Patient-Level Train / Val / Test Split

In [6]:
def patient_split(df, feature_cols, seed=SEED, train_frac=0.8, val_frac=0.1):
    """
    Split by unique patient IDs. Each sequence = (T, F) float32 tensor.
    Returns (train_seqs, val_seqs, test_seqs) and train normalisation stats.
    """
    rids = df['RID'].unique()
    rng  = np.random.default_rng(seed)
    rng.shuffle(rids)
    n = len(rids)
    n_tr = int(n * train_frac); n_va = int(n * val_frac)
    train_rids = set(rids[:n_tr]); val_rids = set(rids[n_tr:n_tr+n_va])
    test_rids  = set(rids[n_tr+n_va:])
    print(f'  Patients → Train:{len(train_rids)}  Val:{len(val_rids)}  Test:{len(test_rids)}')

    train_mask = df['RID'].isin(train_rids)
    mu  = df.loc[train_mask, feature_cols].mean().values.astype(np.float32)
    std = df.loc[train_mask, feature_cols].std().replace(0, 1).values.astype(np.float32)

    def build(rid_set):
        seqs = []
        for rid, grp in df[df['RID'].isin(rid_set)].groupby('RID'):
            grp_s = grp.sort_values('_month') if '_month' in grp.columns else grp
            X = ((grp_s[feature_cols].values.astype(np.float32) - mu) / std)
            X = np.nan_to_num(X, nan=0.0)
            y = int(grp_s['target'].iloc[-1])
            seqs.append((torch.tensor(X), y))
        return seqs

    return build(train_rids), build(val_rids), build(test_rids), mu, std

print('=== FULL split ===')
train_full, val_full, test_full, mu_full, std_full = patient_split(df_full, feat_full)
print('\n=== ALG split ===')
train_alg, val_alg, test_alg, mu_alg, std_alg = patient_split(df_alg, feat_alg)


=== FULL split ===
  Patients → Train:1932  Val:241  Test:243

=== ALG split ===
  Patients → Train:1932  Val:241  Test:243


## 6. By-Domain PCA Reduction

In [7]:
def build_flat_matrix(sequences):
    """Stack all visit feature vectors; track patient structure."""
    Xf, boundaries = [], []
    ptr = 0
    for seq, lbl in sequences:
        X = seq.numpy()
        Xf.append(X)
        boundaries.append((ptr, ptr + len(X)))
        ptr += len(X)
    return np.vstack(Xf), boundaries

def repack_sequences(X_reduced, boundaries, original_seqs):
    """Reconstruct (X_seq, label) list from reduced flat matrix."""
    new_seqs = []
    for (start, end), (_, lbl) in zip(boundaries, original_seqs):
        chunk = X_reduced[start:end].astype(np.float32)
        new_seqs.append((torch.tensor(chunk), lbl))
    return new_seqs

def get_domain_indices(feat_cols, domain_groups):
    """Return {domain: [column_indices]} mapping."""
    col_idx  = {c: i for i, c in enumerate(feat_cols)}
    domain_idx = {}
    assigned = set()
    for domain, keywords in domain_groups.items():
        idxs = []
        for col in feat_cols:
            if any(col == kw or col.startswith(kw + '_') for kw in keywords):
                if col not in assigned:
                    idxs.append(col_idx[col])
                    assigned.add(col)
        if idxs:
            domain_idx[domain] = idxs
    other = [col_idx[c] for c in feat_cols if c not in assigned]
    if other:
        domain_idx['other'] = other
    return domain_idx


def fit_domain_pca(train_seqs, val_seqs, test_seqs, feat_cols,
                   n_per_domain=N_PER_DOMAIN, stream_tag=''):
    """
    Fit PCA independently for each domain, concatenate results.
    Returns: reduced_train, reduced_val, reduced_test, pca_reducers_dict, n_out_total
    pca_reducers_dict is saved for inference.
    """
    domain_idx = get_domain_indices(feat_cols, DOMAIN_GROUPS)
    print(f'[{stream_tag}] Domain PCA | domains: ' +
          ', '.join(f'{d}({len(v)})' for d, v in domain_idx.items()))

    Xtr, b_tr = build_flat_matrix(train_seqs)
    Xva, b_va = build_flat_matrix(val_seqs)
    Xte, b_te = build_flat_matrix(test_seqs)

    parts_tr, parts_va, parts_te = [], [], []
    pca_reducers = {}

    for domain, idxs in domain_idx.items():
        Xd_tr = Xtr[:, idxs]; Xd_va = Xva[:, idxs]; Xd_te = Xte[:, idxs]
        n_comp = min(n_per_domain, len(idxs), Xd_tr.shape[0] - 1)
        pca = PCA(n_components=n_comp, random_state=42)
        parts_tr.append(pca.fit_transform(Xd_tr))
        parts_va.append(pca.transform(Xd_va))
        parts_te.append(pca.transform(Xd_te))
        pca_reducers[domain] = {'pca': pca, 'indices': idxs, 'n_comp': n_comp}

    Xtr_r = np.hstack(parts_tr).astype('float32')
    Xva_r = np.hstack(parts_va).astype('float32')
    Xte_r = np.hstack(parts_te).astype('float32')
    n_out  = Xtr_r.shape[1]
    print(f'  {Xtr.shape[1]}D → {n_out}D (concatenated domain PCA)')

    return (repack_sequences(Xtr_r, b_tr, train_seqs),
            repack_sequences(Xva_r, b_va, val_seqs),
            repack_sequences(Xte_r, b_te, test_seqs),
            pca_reducers, n_out)

print('=== BY-DOMAIN PCA — FULL ===')
train_full_dp, val_full_dp, test_full_dp, pca_full, N_IN_FULL = fit_domain_pca(
    train_full, val_full, test_full, feat_full, stream_tag='FULL')

print('\n=== BY-DOMAIN PCA — ALG ===')
train_alg_dp, val_alg_dp, test_alg_dp, pca_alg, N_IN_ALG = fit_domain_pca(
    train_alg, val_alg, test_alg, feat_alg, stream_tag='ALG')


=== BY-DOMAIN PCA — FULL ===
[FULL] Domain PCA | domains: cognitive(68), functional(33), biomarker(51), demographics(14), medical_history(74), other(60)
  300D → 36D (concatenated domain PCA)

=== BY-DOMAIN PCA — ALG ===
[ALG] Domain PCA | domains: cognitive(68), functional(33), biomarker(51), demographics(14), medical_history(74), other(29)
  269D → 36D (concatenated domain PCA)


## 7. Dataset & DataLoader

In [8]:
class ADNIDataset(Dataset):
    def __init__(self, sequences):
        self.seqs = sequences
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        X, y = self.seqs[i]
        return X, torch.tensor(y, dtype=torch.long)


def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in seqs], dtype=torch.long)
    padded  = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    return padded, torch.stack(labels), lengths


def make_loaders(train_seqs, val_seqs, test_seqs, batch_size=32):
    tr = DataLoader(ADNIDataset(train_seqs), batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    va = DataLoader(ADNIDataset(val_seqs),   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    te = DataLoader(ADNIDataset(test_seqs),  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    return tr, va, te

print("Dataset and loader utilities ready.")


Dataset and loader utilities ready.


## 8. Model Architectures (Temporal Attention + Residual RNN)

In [9]:
class TemporalAttention(nn.Module):
    """Soft attention over time steps, padding-mask aware."""
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hs, lengths):
        B, T, H = hs.shape
        scores = self.attn(hs).squeeze(-1)
        mask   = torch.arange(T, device=hs.device).unsqueeze(0) < lengths.unsqueeze(1)
        scores = scores.masked_fill(~mask, float('-inf'))
        weights = F.softmax(scores, dim=1).unsqueeze(-1)
        context = (weights * hs).sum(dim=1)
        return context, weights.squeeze(-1)


class ResidualRNNBlock(nn.Module):
    """Single-layer RNN/GRU/LSTM with residual projection."""
    def __init__(self, cell_type, input_size, hidden_size, dropout=0.0):
        super().__init__()
        self.cell_type = cell_type
        RNNClass = {'rnn': nn.RNN, 'gru': nn.GRU, 'lstm': nn.LSTM}[cell_type.lower()]
        self.rnn     = RNNClass(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.norm    = nn.LayerNorm(hidden_size)
        self.proj    = nn.Linear(input_size, hidden_size) if input_size != hidden_size else nn.Identity()

    def forward(self, x, hx=None):
        out, hx_new = self.rnn(x, hx)
        residual    = self.proj(x)
        out         = self.norm(self.dropout(out) + residual)
        return out, hx_new


class RecurrentClassifier(nn.Module):
    """
    Stacked residual RNN/GRU/LSTM with temporal attention.
    Used for cfg2: cell_type='rnn', hidden_size=256, num_layers=2, dropout=0.3
    """
    def __init__(self, input_size, num_classes, cell_type='rnn',
                 hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.cell_type  = cell_type.lower()
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.blocks = nn.ModuleList([
            ResidualRNNBlock(self.cell_type, hidden_size, hidden_size, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.attention  = TemporalAttention(hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes)
        )

    def forward(self, x, lengths):
        h = self.input_proj(x)
        for block in self.blocks:
            h, _ = block(h)
        context, _ = self.attention(h, lengths)
        return self.classifier(context)

print("RecurrentClassifier defined ✓")


RecurrentClassifier defined ✓


## 9. Training Utilities

In [10]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_epoch(model, loader, optimiser, criterion):
    model.train()
    total_loss, correct, total = 0., 0, 0
    for X, y, lengths in loader:
        X, y, lengths = X.to(DEVICE), y.to(DEVICE), lengths.to(DEVICE)
        optimiser.zero_grad()
        logits = model(X, lengths)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        total_loss += loss.item() * len(y); correct += (logits.argmax(1) == y).sum().item(); total += len(y)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0., 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for X, y, lengths in loader:
        X, y, lengths = X.to(DEVICE), y.to(DEVICE), lengths.to(DEVICE)
        logits = model(X, lengths)
        loss   = criterion(logits, y)
        probs  = F.softmax(logits, dim=1)
        total_loss += loss.item() * len(y); correct += (logits.argmax(1) == y).sum().item(); total += len(y)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return total_loss/total, correct/total, np.array(all_preds), np.array(all_labels), np.array(all_probs)


def train_model(model, train_loader, val_loader, lr=1e-3, epochs=EPOCHS, patience=PATIENCE,
                num_classes=3, class_weights=None):
    crit = nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE) if class_weights is not None else None
    )
    opt  = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_state, cnt = 0., None, 0
    history = []
    for epoch in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train_loader, opt, crit)
        _, _, preds, labels, probs = evaluate(model, val_loader, crit)
        vf1 = f1_score(labels, preds, average='macro', zero_division=0)
        history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc, 'val_f1': vf1})
        if vf1 > best_f1:
            best_f1, best_state, cnt = vf1, copy.deepcopy(model.state_dict()), 0
        else:
            cnt += 1
        if cnt >= patience:
            print(f'  Early stop at epoch {epoch+1} | best val F1={best_f1:.4f}')
            break
    model.load_state_dict(best_state)
    return model, best_f1, history

# BEFORE (broken)
@torch.no_grad()
def score_model(model, test_seqs, le, stream_tag, model_name):
    te_loader = make_loaders([], [], test_seqs, batch_size=32)[2]  # ← passing [] crashes
    ...

# AFTER (fixed)
@torch.no_grad()
def score_model(model, test_seqs, le, stream_tag, model_name):
    te_loader = DataLoader(ADNIDataset(test_seqs), batch_size=32,
                           shuffle=False, collate_fn=collate_fn)
    ...

@torch.no_grad()
def score_model(model, test_seqs, le, stream_tag, model_name):
    # Build test loader directly — don't pass empty lists to make_loaders
    te_loader = DataLoader(ADNIDataset(test_seqs), batch_size=32,
                           shuffle=False, collate_fn=collate_fn)
    crit = nn.CrossEntropyLoss()
    _, acc, preds, labels, probs = evaluate(model, te_loader, crit)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(np.eye(len(le.classes_))[labels], probs,
                            multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    print(f'  [{stream_tag}] {model_name}: Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
    print(classification_report(labels, preds, target_names=le.classes_, zero_division=0))
    return {'model': model_name, 'feature_set': stream_tag,
            'f1_macro': f1, 'accuracy': acc, 'auc_roc_macro': auc}

print("Training utilities ready.")


Training utilities ready.


## 10. Train RNN cfg2 (Both Streams)

In [11]:
# RNN cfg2 (CORRECT: hidden=128, layers=3, dropout=0.3, lr=3e-4, batch=16 — best from dim-red experiments)
cfg = RNN_CFG2
trained_rnn = {}
results      = []

def train_rnn_cfg2(stream_tag, train_dp, val_dp, test_dp, n_in, le):
    print(f'\n{"="*60}')
    print(f'Training RNN_cfg2 [{stream_tag.upper()}]  input_size={n_in}')
    print(f'{"="*60}')
    tr, va, te = make_loaders(train_dp, val_dp, test_dp, batch_size=cfg['batch_size'])

    model = RecurrentClassifier(
        input_size=n_in, num_classes=len(le.classes_),
        cell_type='rnn', hidden_size=cfg['hidden_size'],
        num_layers=cfg['num_layers'], dropout=cfg['dropout']
    ).to(DEVICE)
    print(f'  Parameters: {count_params(model):,}')

    # Class weights from training labels
    train_labels = [lbl for _, lbl in train_dp]
    cw = None
    if len(set(train_labels)) > 1:
        from sklearn.utils.class_weight import compute_class_weight
        cw_arr = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
        cw = cw_arr.tolist()

    model, best_f1, history = train_model(model, tr, va, lr=cfg['lr'],
                                          num_classes=len(le.classes_), class_weights=cw)
    r = score_model(model, test_dp, le, stream_tag, 'RNN_cfg2_domain_pca')
    trained_rnn[stream_tag] = model
    results.append(r)
    return model, best_f1

model_full, f1_full = train_rnn_cfg2('full', train_full_dp, val_full_dp, test_full_dp, N_IN_FULL, le_full)
model_alg,  f1_alg  = train_rnn_cfg2('alg',  train_alg_dp,  val_alg_dp,  test_alg_dp,  N_IN_ALG,  le_alg)

print(f'\nFull stream → best val F1: {f1_full:.4f}')
print(f'ALG  stream → best val F1: {f1_alg:.4f}')



Training RNN_cfg2 [FULL]  input_size=36
  Parameters: 113,155
  Early stop at epoch 19 | best val F1=0.7697
  [full] RNN_cfg2_domain_pca: Acc=0.8272  F1=0.7829  AUC=0.9459
                precision    recall  f1-score   support

            CN       0.94      0.88      0.91       144
MCI_converting       0.81      0.81      0.81        52
    MCI_stable       0.58      0.70      0.63        47

      accuracy                           0.83       243
     macro avg       0.78      0.79      0.78       243
  weighted avg       0.84      0.83      0.83       243


Training RNN_cfg2 [ALG]  input_size=36
  Parameters: 113,155
  Early stop at epoch 22 | best val F1=0.8103
  [alg] RNN_cfg2_domain_pca: Acc=0.8354  F1=0.7938  AUC=0.9488
                precision    recall  f1-score   support

            CN       0.92      0.90      0.91       144
MCI_converting       0.89      0.75      0.81        52
    MCI_stable       0.59      0.74      0.66        47

      accuracy                     

## 11. Results Summary

In [12]:
df_res = pd.DataFrame(results)
print(df_res[['feature_set','model','accuracy','f1_macro','auc_roc_macro']].round(4).to_string(index=False))


feature_set               model  accuracy  f1_macro  auc_roc_macro
       full RNN_cfg2_domain_pca    0.8272    0.7829         0.9459
        alg RNN_cfg2_domain_pca    0.8354    0.7938         0.9488


## 12. Save All Artifacts

In [13]:
def save_stability_stream(stream_tag, model, pca_reducers, feat_cols,
                          mu, std, le_target, cat_le_map, global_means):
    out_dir = f'{SAVE_ROOT}/{stream_tag}'
    os.makedirs(out_dir, exist_ok=True)

    # Neural model state dict
    torch.save(model.state_dict(), f'{out_dir}/rnn_cfg2_domain_pca.pt')
    # Save model config for easy reload
    model_cfg = {
        'input_size':  next(iter(pca_reducers.values()))['n_comp'] * len(pca_reducers),
        'num_classes': len(le_target.classes_),
        'cell_type':   'rnn',
        'hidden_size': RNN_CFG2['hidden_size'],
        'num_layers':  RNN_CFG2['num_layers'],
        'dropout':     RNN_CFG2['dropout'],
    }
    # Compute actual input size from sum of PCA components
    model_cfg['input_size'] = sum(v['n_comp'] for v in pca_reducers.values())
    joblib.dump(model_cfg, f'{out_dir}/model_config.pkl')

    # PCA reducers (domain → {'pca', 'indices', 'n_comp'})
    joblib.dump(pca_reducers, f'{out_dir}/pca_reducers.pkl')

    # Normalisation stats (mu/std computed from training set before PCA)
    joblib.dump({'mu': mu, 'std': std}, f'{out_dir}/norm_stats.pkl')

    # Preprocessing objects
    joblib.dump(feat_cols,    f'{out_dir}/feature_cols.pkl')
    joblib.dump(le_target,    f'{out_dir}/label_encoder.pkl')
    joblib.dump(cat_le_map,   f'{out_dir}/cat_encoders.pkl')
    joblib.dump(global_means, f'{out_dir}/global_means.pkl')

    # Results CSV
    pd.DataFrame(results).to_csv(f'{out_dir}/results.csv', index=False)
    print(f"  [{stream_tag}] Saved all artifacts → {out_dir}/")

save_stability_stream('full', model_full, pca_full, feat_full, mu_full, std_full,
                      le_full, cat_le_full, gmeans_full)
save_stability_stream('alg',  model_alg,  pca_alg,  feat_alg,  mu_alg,  std_alg,
                      le_alg,  cat_le_alg,  gmeans_alg)
print("\nAll stability model artifacts saved!")


  [full] Saved all artifacts → ./stability_models/full/
  [alg] Saved all artifacts → ./stability_models/alg/

All stability model artifacts saved!


## 13. Inference Functions — Use After Loading Saved Models

In [14]:
# ─────────────────────────────────────────────────────────────────────────
# LOADING & INFERENCE (run in a fresh session to do inference)
# ─────────────────────────────────────────────────────────────────────────

def load_stability_artifacts(stream_tag, save_root='./stability_models'):
    """Load all saved artifacts for inference. Returns a dict."""
    out_dir = f'{save_root}/{stream_tag}'
    arts = {}
    arts['pca_reducers']  = joblib.load(f'{out_dir}/pca_reducers.pkl')
    arts['norm_stats']    = joblib.load(f'{out_dir}/norm_stats.pkl')
    arts['feature_cols']  = joblib.load(f'{out_dir}/feature_cols.pkl')
    arts['label_encoder'] = joblib.load(f'{out_dir}/label_encoder.pkl')
    arts['cat_encoders']  = joblib.load(f'{out_dir}/cat_encoders.pkl')
    arts['global_means']  = joblib.load(f'{out_dir}/global_means.pkl')
    arts['model_config']  = joblib.load(f'{out_dir}/model_config.pkl')
    return arts


def load_stability_model(stream_tag, arts, save_root='./stability_models', device_inf=None):
    """Rebuild model from config and load saved weights."""
    if device_inf is None:
        device_inf = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cfg = arts['model_config']
    model = RecurrentClassifier(
        input_size=cfg['input_size'], num_classes=cfg['num_classes'],
        cell_type=cfg['cell_type'], hidden_size=cfg['hidden_size'],
        num_layers=cfg['num_layers'], dropout=cfg['dropout']
    ).to(device_inf)
    state = torch.load(f'{save_root}/{stream_tag}/rnn_cfg2_domain_pca.pt',
                       map_location=device_inf)
    model.load_state_dict(state)
    model.eval()
    return model


def preprocess_visits_for_inference(visit_list, arts):
    """
    Convert a list of visit dicts into a PCA-reduced (T, D) float32 array,
    ready to feed into the model.

    visit_list: list of dicts [{feature_name: value}, ...]
                ordered chronologically (earliest first)
    arts: output of load_stability_artifacts()
    Returns: torch.FloatTensor of shape (1, T, D) and lengths tensor
    """
    feat_cols   = arts['feature_cols']
    cat_enc     = arts['cat_encoders']
    global_means= arts['global_means']
    mu, std     = arts['norm_stats']['mu'], arts['norm_stats']['std']
    pca_red     = arts['pca_reducers']

    rows = []
    for visit in visit_list:
        row = []
        for col in feat_cols:
            if col in cat_enc:
                val = visit.get(col, 'UNKNOWN')
                val = str(val) if not pd.isna(val) else 'UNKNOWN'
                le  = cat_enc[col]
                if val in le.classes_:
                    row.append(float(le.transform([val])[0]))
                else:
                    row.append(float(le.transform([le.classes_[0]])[0]))
            elif col.endswith('_missing'):
                base = col[:-len('_missing')]
                row.append(1.0 if (base not in visit or pd.isna(visit.get(base))) else 0.0)
            elif col.endswith('_delta'):
                # Delta can't be computed for single visits; use 0
                row.append(0.0)
            else:
                val = visit.get(col, np.nan)
                if pd.isna(val):
                    val = float(global_means.get(col, 0.0))
                row.append(float(val))
        rows.append(row)

    X_raw = np.array(rows, dtype=np.float32)  # (T, F)
    X_norm = np.nan_to_num((X_raw - mu) / std, nan=0.0)

    # Apply domain PCA
    parts = []
    for domain, red in pca_red.items():
        idxs = red['indices']
        valid = [i for i in idxs if i < X_norm.shape[1]]
        if not valid: continue
        parts.append(red['pca'].transform(X_norm[:, valid]))

    X_pca = np.hstack(parts).astype(np.float32)  # (T, D)
    X_t   = torch.tensor(X_pca).unsqueeze(0)      # (1, T, D)
    lengths = torch.tensor([len(visit_list)], dtype=torch.long)
    return X_t, lengths


def predict_patient_stability(visit_list, arts, model, device_inf=None):
    """
    Predict MCI stability for a patient given their longitudinal visit history.

    visit_list: [{'MMSCORE': 24, 'age': 72, ...}, {...}, ...]  # one dict per visit
    arts:  output of load_stability_artifacts()
    model: loaded RecurrentClassifier (output of load_stability_model())

    Returns: dict with prediction, probabilities, and class names
    """
    if device_inf is None:
        device_inf = next(model.parameters()).device
    model.eval()
    X, lengths = preprocess_visits_for_inference(visit_list, arts)
    X = X.to(device_inf); lengths = lengths.to(device_inf)
    with torch.no_grad():
        logits = model(X, lengths)
        probs  = F.softmax(logits, dim=1).cpu().numpy()[0]
    pred  = int(np.argmax(probs))
    le    = arts['label_encoder']
    label = le.inverse_transform([pred])[0]
    return {
        'prediction':    label,
        'class_index':   pred,
        'probabilities': {le.classes_[i]: float(p) for i, p in enumerate(probs)},
        'class_names':   list(le.classes_),
        'n_visits_used': len(visit_list),
    }

print("Inference functions ready.")


Inference functions ready.


## 14. Example Inference (Demo)

In [15]:
# ── Demo: run inference on a test patient ───────────────────────────────
arts_alg_loaded  = load_stability_artifacts('alg')
model_alg_loaded = load_stability_model('alg', arts_alg_loaded)

# Build demo visit list from the first test patient's sequence
first_seq, first_lbl = test_alg_dp[0]
feat_cols_alg = arts_alg_loaded['feature_cols']

# The sequences are already PCA-reduced so we demo with a fake raw visit dict
demo_visit = {col: 0.0 for col in feat_cols_alg}   # placeholder — replace with real values
demo_visits = [demo_visit] * first_seq.shape[0]     # replicate for each visit

result = predict_patient_stability(demo_visits, arts_alg_loaded, model_alg_loaded)
print("Prediction result:", result)
print()
print("True label:", le_alg.inverse_transform([first_lbl])[0])

# ── For real inference ───────────────────────────────────────────────────
# arts  = load_stability_artifacts('alg')  # or 'full'
# model = load_stability_model('alg', arts)

# patient_visits = [
#     {'MMSCORE': 26, 'MOCA': 25, 'FAQ': 2, 'Abeta42': 180.0, 'age': 68, ...},  # visit 1
#     {'MMSCORE': 24, 'MOCA': 23, 'FAQ': 4, 'Abeta42': 165.0, 'age': 69, ...},  # visit 2
#     ...
# ]
# result = predict_patient_stability(patient_visits, arts, model)
# print(result)


Prediction result: {'prediction': 'MCI_stable', 'class_index': 2, 'probabilities': {'CN': 0.0036451166961342096, 'MCI_converting': 0.09016147255897522, 'MCI_stable': 0.9061934351921082}, 'class_names': ['CN', 'MCI_converting', 'MCI_stable'], 'n_visits_used': 7}

True label: CN
